# Analyze a Baofeng Transmission

This notebook measures deviation, CTCSS energy, and spectral occupancy from a local transmission capture if available. Without a capture, it uses a synthetic stand-in that behaves like a narrowband FM handheld transmission.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
CAPTURE_ROOT = ROOT / "assets" / "local"
capture_status = probe_rtlsdr()
display(Markdown(
    f"**RTL-SDR status:** installed={capture_status['installed']}, "
    f"available={capture_status['available']}. {capture_status['message']}"
))


In [ ]:
capture_path = CAPTURE_ROOT / "baofeng_tx_iq.npz"
if capture_path.exists():
    fs_iq, iq = load_complex_capture(capture_path)
    print(f"Loaded local capture: {capture_path.name}, fs={fs_iq}")
else:
    fs_iq = 240_000
    t = np.arange(0, 3.0, 1 / fs_iq)
    voice = normalize(0.7 * np.sin(2 * np.pi * 1000 * t) + 0.2 * np.sin(2 * np.pi * 200 * t))
    composite = normalize(voice + 0.15 * np.cos(2 * np.pi * 141.3 * t))
    iq = synthesize_fm_iq(composite, fs=fs_iq, carrier_offset=35_000, freq_dev=2500)
    print("Using synthetic handheld-transmission fallback.")


In [ ]:
baseband = complex_mix_down(iq, fs_iq, 35_000)
audio = fm_demodulate_iq(baseband, fs=fs_iq, audio_cutoff=4000)
inst_freq = np.angle(baseband[1:] * np.conj(baseband[:-1])) * fs_iq / (2 * np.pi)
peak_dev = np.max(np.abs(inst_freq - np.mean(inst_freq)))

freqs, spectrum = power_spectrum(audio, fs_iq, nfft=8192)
ctcss_mask = (freqs >= 60) & (freqs <= 260)
ctcss_freq = freqs[ctcss_mask][np.argmax(spectrum[ctcss_mask])]
occupied_bw = freqs[np.where(spectrum > (np.max(spectrum) - 26))[0][-1]]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
plot_spectrum(np.real(baseband), fs=fs_iq, ax=axes[0], title="Baseband spectrum")
axes[0].set_xlim(0, 50_000)
axes[0].set_ylim(-100, 5)
plot_spectrum(audio, fs=fs_iq, ax=axes[1], title="Recovered audio spectrum")
axes[1].set_xlim(0, 4000)
axes[1].set_ylim(-100, 5)
axes[2].hist(inst_freq - np.mean(inst_freq), bins=100, color="tab:orange")
axes[2].set_title("Instantaneous frequency spread")
axes[2].set_xlabel("Deviation (Hz)")
plt.tight_layout()

display(Markdown(f"**Estimated peak deviation:** {peak_dev:.0f} Hz"))
display(Markdown(f"**Strongest sub-audible component:** {ctcss_freq:.1f} Hz"))
display(Markdown(f"**Approximate occupied audio-side bandwidth at -26 dB:** {occupied_bw:.0f} Hz"))


## Key Takeaway

Compliance questions turn into measurements once you have a capture: estimate deviation from instantaneous frequency, find tone energy in the low audio band, and inspect occupied bandwidth against your assumptions.